# Explicabilidad de ConvNeXt-Tiny mediante SHAP

Este notebook reproduce el análisis presentado en `paper.tex`. Se carga el modelo `facebook/convnext-tiny-224` desde Hugging Face, se clasifican tres fotografías de gatos y se calculan los valores SHAP correspondientes a las clases con mayor probabilidad predicha para cada imagen.

Requiere las dependencias listadas en `requirements.txt` (torch, transformers, shap, pillow, matplotlib, numpy).

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import shap
from transformers import AutoImageProcessor, AutoModelForImageClassification

ASSETS = os.path.join("..", "assets")
FIGURES = os.path.join("..", "figures")

IMAGES = [
    ("gato_a_tuxedo.jpg", "Gato A (tuxedo)"),
    ("gato_b_atigrado_blanco.jpg", "Gato B (atigrado/blanco)"),
    ("gato_c_atigrado_solido.jpg", "Gato C (atigrado solido)"),
]

MODEL_NAME = "facebook/convnext-tiny-224"

## Carga del modelo y preprocesamiento

Se usa `AutoImageProcessor` para obtener el tamaño de entrada y los parámetros de normalización esperados por el modelo, y `AutoModelForImageClassification` para obtener el modelo en modo de evaluación.

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(MODEL_NAME)
model.eval()

size = processor.size.get("shortest_edge", processor.size.get("height", 224))
mean = torch.tensor(processor.image_mean).view(1, 3, 1, 1)
std = torch.tensor(processor.image_std).view(1, 3, 1, 1)

def load_image(path, sz):
    img = Image.open(path).convert("RGB").resize((sz, sz))
    return np.array(img)

images = np.stack([load_image(os.path.join(ASSETS, fname), size) for fname, _ in IMAGES])
images.shape

## Función de predicción

Se define una función que recibe un lote de imágenes como arreglo numpy (uint8, `[N, H, W, 3]`) y devuelve las probabilidades de clase. Esta función se pasa directamente a `shap.Explainer`.

No se utiliza el `pipeline` de `transformers` como modelo para SHAP, ya que el wrapper `shap.models.TransformersPipeline` está diseñado para pipelines de texto y falla al recibir arreglos de imagen en formato numérico.

In [ ]:
def predict(x):
    x = torch.tensor(x).permute(0, 3, 1, 2).float() / 255.0
    x = (x - mean) / std
    with torch.no_grad():
        logits = model(pixel_values=x).logits
        probs = F.softmax(logits, dim=1)
    return probs.numpy()

id2label = model.config.id2label
output_names = [id2label[i] for i in range(len(id2label))]

## Predicciones top-5

In [ ]:
probs_all = predict(images)
for (fname, label), probs in zip(IMAGES, probs_all):
    top5 = np.argsort(probs)[::-1][:5]
    print(f"\n{label} ({fname}):")
    for idx in top5:
        print(f"  {id2label[idx]}: {probs[idx]:.4f}")

## Cálculo de valores SHAP

Se enmascara la imagen mediante inpainting (`inpaint_telea`) y se calculan los valores SHAP para las tres clases con mayor probabilidad predicha en cada imagen. El parámetro `max_evals` controla cuántas evaluaciones del modelo se realizan por imagen; se mantiene en un valor moderado para que el cálculo sea viable en CPU.

In [ ]:
masker = shap.maskers.Image("inpaint_telea", images[0].shape)
explainer = shap.Explainer(predict, masker, output_names=output_names)

## Visualización y exportación de figuras

Cada imagen se explica por separado, en un lote de tamaño uno. Al pasar las tres imágenes juntas con clases top-k distintas por fila, la versión instalada de `shap` no logra reconstruir la dimensionalidad de salida al graficar, por lo que explicar una imagen a la vez evita ese problema.

`shap.image_plot` genera, para cada imagen, una fila con la imagen original seguida de una columna por cada una de las tres clases explicadas. Los tonos cálidos (rosado) indican regiones que incrementan la probabilidad de la clase, y los tonos fríos (azul) indican regiones que la reducen.

In [ ]:
import matplotlib.pyplot as plt

for i, (fname, label) in enumerate(IMAGES):
    print(f"Procesando {label}...")
    sv = explainer(
        images[i:i+1],
        max_evals=300,
        batch_size=50,
        outputs=shap.Explanation.argsort.flip[:3],
    )
    shap.image_plot(sv, show=False)
    out_path = os.path.join(FIGURES, f"shap_{fname.replace('.jpg', '')}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print("Guardado:", out_path)